# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Add src to path so we can import minutia modules
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../../src').resolve()))


In [ ]:
import datetime
import pathlib

import isodate
from dotenv import load_dotenv


def get_isodatetime():
    """get the current time as an iso datetime"""
    current = datetime.datetime.now(tz=datetime.timezone.utc)
    iso_value = isodate.datetime_isoformat(current)
    return iso_value

def get_isodate():
    """get the current time as an iso datetime"""
    current = datetime.datetime.now(tz=datetime.timezone.utc)
    iso_value = isodate.date_isoformat(current)
    return iso_value

#local functions
from minutia.steps import video_extraction, audio_transcription, transcript_merger, summarize_transcript
from minutia.logging import logger


# Load environment variables
load_dotenv()

## Setup

In [ ]:
output_dir = pathlib.Path('../../+outputs')
recording_dir = pathlib.Path('../../+recordings')
# Analyze Recordings directory
logger.info(f"Recording dir: {recording_dir.resolve()}")
mkv_filepaths = list(recording_dir.glob("*.mkv"))
if len(mkv_filepaths) == 0:
    logger.error(f"No *.mkv files found in '{recording_dir.resolve()}'")
else:
    logger.info(f"Found {len(mkv_filepaths)} *.mkv files:")
    for p in mkv_filepaths:
        logger.info(f"   '../{recording_dir.stem}/{str(p.name)}'")

recording_filepath = recording_dir / 'Example-Song+MLKSpeech.mkv'

if len(mkv_filepaths)==0:
    recording_filepath = None
    logger.error(f"No recording file selected and no files found.")

elif recording_filepath not in mkv_filepaths:
    recording_filepath = mkv_filepaths[0]
    logger.info(f"Could not find selected file, defaulting to first found:")
    logger.info(f"    '../{recording_dir.stem}/{recording_filepath.name}'")
else:
    logger.info(f"Selected *.mkv File:")
    logger.info(f"    '../{recording_dir.stem}/{recording_filepath.name}'")

# Setup Output Directory
logger.info(f"Output dir: {output_dir.resolve()}")
if output_dir.exists() is False:
    output_dir.mkdir(parents=True, exist_ok=True)
    logger.info(f"    Created output dir:")
logger.info(f"    '{output_dir.resolve()}")
    

In [ ]:
logger.info(f"Extracting Audio...")
logger.info(f"    recording: '{recording_filepath.name}'")
logger.info(f"    output_dir: '{output_dir.resolve()}'")
try:
   result = video_extraction.process_mkv_file(str(recording_filepath), str(output_dir))
except Exception as e:
    logger.error(f"An error occurred during the extraction process: {str(e)}")

In [ ]:
microphone_audio_path = output_dir / "microphone_audio.wav"
desktop_audio_path = output_dir / "desktop_audio.wav"


logger.info(f"Transcribing Audio...")
logger.info(f"    microphone-track: '{microphone_audio_path}'")
logger.info(f"    desktop-track: '{desktop_audio_path}'")

try:
    filenames = audio_transcription.process_audio_files(
        mic_audio_path=str(microphone_audio_path),
        desktop_audio_path=str(desktop_audio_path),
        output_dir=str(output_dir),
        whisper_model_name = 'medium.en',
        use_gpu=True,
    )
    
    logger.info(f"Transcribed Files:")
    for name, path in filenames.items():
        logger.info(f"    {name}: '{path}'")
except Exception as e:
    logger.error(f"An error occurred during the transcription process: {str(e)}")

In [ ]:
mic_transcript_path = filenames['microphone']
desktop_transcript_path = filenames['desktop']
transcript_file_path = output_dir / "merged_transcript.json"


logger.info(f"Merging Transcripts...")
logger.info(f"    microphone-transcript: '{mic_transcript_path}'")
logger.info(f"    desktop-transcript: '{desktop_transcript_path}'")

try:
    transcript_json = transcript_merger.process_transcripts(
        mic_transcript_file=str(mic_transcript_path),
        desktop_transcript_file=str(desktop_transcript_path),
        output_file=str(transcript_file_path),
        parse_overlaps=False
    )
except Exception as e:
    logger.error(f"An error occurred during the merging process: {str(e)}")

logger.info(f"Merged Transcript: '{transcript_file_path}'") 

In [ ]:
summarize_model_name = 'claude-4-5-sonnet'

logger.info(f"Summarizing Meeting...")

transcript_data = summarize_transcript.load_transcript(str(transcript_file_path))
transcript_text = summarize_transcript.prepare_transcript_for_summary(transcript_data)
total_tokens, estimated_price = summarize_transcript.estimate_tokens_and_price(
    transcript_text,
    model=summarize_model_name
)
print(transcript_text)


# Summarize

In [ ]:
def generate_meeting_summary_prompt(markdown_notes:str="", personal_info:str=""):
    prompt = f"""# 🧙‍♂️ Comprehensive and Detailed Meeting Insight Generator

#CONTEXT: Your task is to create a thorough and detailed summary of a meeting transcript. The summary should be comprehensive, capturing the nuances of discussions, technical details, and decisions made. The target audience may not have been present at the meeting but requires an in-depth understanding of the topics discussed.

#ROLE: You are an expert meeting analyst and technical communicator with a keen eye for detail. Your goal is to distill complex discussions into a structured, informative summary that provides significant value to both attendees and non-attendees.

#RESPONSE GUIDELINES:
1. Begin with an executive summary (who, what, where, when) that provides a high-level overview of the meeting's key points.
2. Analyze the transcript and determine the appropriate number of major topics discussed. Create a numbered list of these topics.
3. For each major topic:
   a. Create a detailed heading
   b. Provide an in-depth expansion on technical content, quoting relevant parts of the transcript where appropriate
   c. Describe existing or proposed projects/efforts in detail
   d. Outline decisions made, including the reasoning behind them
   e. Specify timelines and next steps
   f. Integrate relevant personal notes, highlighting them distinctly
   g. Analyze potential implications or impacts of the discussed items
4. Review and add any additional relevant information, including context from your knowledge base if applicable
5. Conclude with a comprehensive list of action items, responsibilities, and overall meeting outcomes
6. Create a section specifically addressing personal notes not covered in the main topics, explaining their context and importance

#TASK CRITERIA:
- Use markdown formatting for headers, lists, and emphasis
- Aim for a highly detailed summary, typically 3-6 pages of content
- Focus on technical accuracy and clarity, providing examples or elaborations where necessary
- Ensure the summary is self-contained and understandable without additional context
- Highlight personal notes using blockquotes and the 📝 emoji
- Prioritize addressing all personal notes, ensuring they are contextually integrated
- The number of major topics should be determined based on the meeting's content and complexity
- Use code blocks for any technical snippets or data presented in the meeting

#INFORMATION ABOUT ME: 
{personal_info}

#PERSONAL NOTES:
{markdown_notes}

#OUTPUT:
Provide the detailed comprehensive summary in the following structure:

## Executive Summary

[Comprehensive overview, including high-priority personal notes and key decisions]

## Major Topics Discussed

1. [Topic 1]
2. [Topic 2]
3. [Topic 3]
... (continue as needed based on the meeting's content)

## Detailed Discussion

### [Topic 1]

[In-depth technical details, projects, decisions, timelines, and analysis]

> 📝 Personal Note: [Relevant personal note integrated and highlighted]
> * [Related bullet point with additional context]
> * [Related bullet point with additional context]

[Additional content addressing the personal note, including potential implications]

[Relevant quotes from the transcript]

### [Topic 2]

[Same structure as Topic 1, with different content]

... (continue for all identified topics)

## Additional Notes and Context

[Any relevant information not covered in the main sections, including personal notes that didn't fit elsewhere and additional context from your knowledge base]

## Action Items and Conclusion

[Comprehensive list of action items, responsibilities, deadlines, and overall meeting outcomes]

## Personal Notes Summary

[A detailed section addressing any remaining personal notes that weren't integrated into the main summary, explaining their context, importance, and potential future implications]

Please summarize the meeting transcript based on the instructions given above, integrating and highlighting the provided personal notes throughout the summary. Ensure that the summary is as detailed and comprehensive as possible, capturing the full essence of the meeting discussions.

If the summary exceeds the token limit, please end your response with '[CONTINUE]' and I will prompt you to continue in a new message.
"""
    return prompt

def extract_transcript_text(transcript_data):
    """
    Extract text from the JSON transcript, combine it into a single string,
    including speaker and timestamp, and ensuring chronological order.
    """
    transcript_entries = transcript_data["transcript"]
    # Sort entries by start_time to ensure chronological order
    sorted_entries = sorted(transcript_entries, key=lambda x: x["start_time"])

    formatted_lines = []
    for entry in sorted_entries:
        speaker = entry["source"].capitalize()
        start_time = f"{entry['start_time']:.2f}"
        text = entry["text"].strip()
        formatted_line = f"[{speaker} {start_time}] {text}"
        formatted_lines.append(formatted_line)

    return "\n".join(formatted_lines)

def get_filename_prompt(content):
    current_date = get_isodate()
    prompt = f"""Please read the <content> completely and provide a filename that can be used to write the file to disk as a markdown. The filename should be concise but also descriptive so its clear what the content contains.

    <filenamerules>
    - only use lowercase letters, numbers, and underscores or dashes
    - prefer underscores where possible as they are operating system independent
    - make the filename convenient for all operating systems (windows, macOS, linux)
    - include the current date in the filename as YYYMMDD (e.g. July 4th , 2024 would be 20240704)
    - include a concise but descriptive name related to the <content>
    - keep the filename as short as possible while containing a useful name and date
    - DONOT include the suffix of the filename and make sure their are no period characters in the filename
    - the current date is {current_date}
    </filenamerules>

    <content>
    {content}
    </content>

    remember ONLY return the filename as a string and follow the <filenamerules>
    """
    return prompt

def get_filename_from_content(content, model_name='claude-4-5-sonnet'):
    """obtain a good filename from content and date"""
    from anthropic import Anthropic
    from minutia.config import resolve_anthropic_api_key
    anthropic_models = summarize_transcript.anthropic_models
    if model_name not in anthropic_models:
        raise KeyError(f"model must be one of [{list(anthropic_models.keys())}]")
    model = anthropic_models.get(model_name)['name']
    client = Anthropic(api_key=resolve_anthropic_api_key())
    prompt = get_filename_prompt(content)

    try:
        response = client.messages.create(
            model=model,
            max_tokens=100,
            temperature=0.7,
            messages=[
                {"role": "user", "content": prompt}
            ],
        )
        return response.content[0].text
    except Exception as e:
        logger.error(f"Error generating summary: {str(e)}")
        raise

def create_markdown_report(content, output_path):
    """reads content and comes up with a good name, then writes to markdown"""
    filename = get_filename_from_content(content, model_name='claude-3-5-sonnet')
    filepath = output_path / f"{filename}.md"
    logger.info(f"Writing markdown file to {filepath}")
    with open(filepath, 'w', encoding='utf-8') as file:
        file.write(content)
    return filepath


In [ ]:
# Main workflow
personal_info="""
Product Manager and senior developer with a background in data engineering, chemical engineering, and modeling.
- expert in python software development and libaries
- extensive familiarity with GCP and cloud computing paradigms

Checking in to a discovery meeting with a client.
"""

markdown_notes = """
<redacted: per-meeting notes and speaker mapping>
"""
system_prompt = generate_meeting_summary_prompt(markdown_notes,personal_info)
transcript_text = extract_transcript_text(transcript_data)

logger.info("Summarizing notes and transcript now...")

try:
    summary = summarize_transcript.generate_long_summary(
        transcript_text, 
        model_name='claude-4-5-sonnet', 
        system_prompt=system_prompt
    )
    filepath = create_markdown_report(content=summary, output_path=output_dir)
    logger.info(f"Summary generated and saved to {filepath}")
except Exception as e:
    logger.error(f"An error occurred during summarization: {str(e)}")
logger.info("Completed Summarization!")

# Whisper Model Testing

Determine what whisper model to use

In [ ]:
import torch
import whisper
import time
import psutil
import GPUtil
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Check GPU availability
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
    
    # Current GPU memory usage
    gpu = GPUtil.getGPUs()[0]
    print(f"Current GPU Memory Used: {gpu.memoryUsed:.0f} MB / {gpu.memoryTotal:.0f} MB")
else:
    print("No GPU detected! Falling back to CPU")